# ARDY auf Kaggle-GPU — echte Text→Motion-Clips für Trainrobot erzeugen

NVIDIA ARDY (SIGGRAPH 2026) erzeugt Motion für das Unitree-G1-Skelett — genau die 36-Spalten-QPOS-Konvention, die Trainrobot 1:1 importiert.

**Ablauf:** 1) GPU aktivieren (Settings → Accelerator → GPU T4) · 2) Zellen der Reihe nach laufen lassen · 3) CSVs erscheinen in `/kaggle/working/ardy_g1/` · 4) Output herunterladen und in den Datensatz (`ardy_g1/`) hochladen oder direkt in der App per „.csv (ARDY)“ importieren.

Hinweis: ARDY braucht den Text-Encoder Llama-3-8B (gated). Falls der HF-Zugriff fehlschlägt, setzt `TEXT_ENCODER_DEVICE=cpu` (Zelle 3 macht das schon) — auf T4 läuft ARDY dann trotzdem.

In [ ]:
# 1) GPU prüfen
!nvidia-smi -L || echo 'KEIN GPU — Settings → Accelerator → GPU T4 aktivieren!'

In [ ]:
# 2) ARDY installieren (Apache-2.0, github.com/nv-tlabs/ardy)
!git clone https://github.com/nv-tlabs/ardy.git /kaggle/working/ardy
%cd /kaggle/working/ardy
!pip install -e . -q

In [ ]:
# 3) Text-Encoder auf CPU (T4-schonend) + ARDY-G1-Checkpoints laden
import os
os.environ['TEXT_ENCODER_DEVICE'] = 'cpu'
!python scripts/download_checkpoints.py --model g1  # oder: huggingface-cli download nvidia/ARDY-G1-RP-25FPS-Horizon52/8

In [ ]:
# 4) Motion erzeugen — eigene Prompts eintragen!
import os
os.makedirs('/kaggle/working/ardy_g1', exist_ok=True)
PROMPTS = [
    ('mein_spaziergang', 'a person walks forward calmly'),
    ('mein_tanz', 'a person dances happily in place'),
    ('mein_sprung', 'a person jumps up energetically'),
]
for name, prompt in PROMPTS:
    !python scripts/generate.py "{prompt}" --model g1 --duration 8 --output /kaggle/working/ardy_g1/{name}.csv
print('Fertig — CSVs liegen in /kaggle/working/ardy_g1/')

## Weiterverwenden
- **Direkt in der App:** Output-Zip herunterladen → Trainrobot → GLB-Leiste → „.csv (ARDY)“ (G1-Roboter aktiv)
- **In den Datensatz:** CSVs (36 Spalten, kein Header) in `ardy_g1/` des Kaggle-Datensatzes `rudolfbewer/trainrobot-motionclips` ergänzen — Version neu hochladen. Die App lädt den Datensatz automatisch.
- Die App erkennt ARDY-CSVs am Format (root + quat wxyz + 29 G1-DoF, MuJoCo-Konvention) und macht daraus automatisch Geist, BC- und PPO-Tracking-Lehrer.